In [1]:
import warnings
warnings.filterwarnings("ignore")
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv




### Data Ingestion

In [2]:
def data_ingestion(sample_txt_file):

    # Text Loader
    from langchain_community.document_loaders import TextLoader
    text_loader = TextLoader(sample_txt_file, encoding="utf-8")
    text_loader
    # Load the document
    document = text_loader.load() # Gives the document structure with page_content and updated metadata
    print(type(document))
    return document
    



# Creating a txt file with the content of the document
# os.makedirs("../data/test_files", exist_ok=True)
sample_txt_file = "quantum_entanglement.txt"
doc = data_ingestion(sample_txt_file)

<class 'list'>


### Chunking


In [3]:
def chunk(document , chunk_size = 500 , chunk_overlap = 50):
    # Split for better RAG performance
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size , chunk_overlap=chunk_overlap , length_function=len,separators=["---", "\n\n", "\n", ". ", " "])
    split_doc = text_splitter.split_documents(document) # split_documents takes a list of documents and splits each document into smaller chunks based on the specified chunk_size and chunk_overlap. The resulting split_doc is a list of smaller Document objects, each containing a portion of the original document's content and metadata.
    print(f"Number of chunks created: {len(split_doc)}")  
    if split_doc:
        print("Example chunk:")
        print(f"Chunk content: {split_doc[0].page_content}") # page_content contains the actual text content of the chunk, which is a portion of the original document's content. This is the part that will be used for processing and retrieval in RAG.
        print(f"Chunk metadata: {split_doc[0].metadata}") # metadata contains the additional information about the chunk, such as its source, author, and creation date. This metadata is inherited from the original document and can be useful for organizing and retrieving chunks later on.
    return split_doc

chunks = chunk(doc)
chunks
        


Number of chunks created: 55
Example chunk:
Chunk content: QUANTUM ENTANGLEMENT: A COMPREHENSIVE OVERVIEW

INTRODUCTION
---------
Chunk metadata: {'source': 'quantum_entanglement.txt'}


[Document(metadata={'source': 'quantum_entanglement.txt'}, page_content='QUANTUM ENTANGLEMENT: A COMPREHENSIVE OVERVIEW\n================================================\n\nINTRODUCTION\n---------'),
 Document(metadata={'source': 'quantum_entanglement.txt'}, page_content='---------\nQuantum entanglement is one of the most fascinating and counterintuitive phenomena in quantum mechanics. It describes a physical phenomenon where two or more particles become correlated in such a way that the quantum state of each particle cannot be described independently of the others, even when separated by large distances. Albert Einstein famously referred to this as "spooky action at a distance," a term he used to express his skepticism about the concept.\n\nHISTORICAL BACKGROUND'),
 Document(metadata={'source': 'quantum_entanglement.txt'}, page_content='------------------'),
 Document(metadata={'source': 'quantum_entanglement.txt'}, page_content='---\nThe concept of quantum entanglement emerged from t

### Embedding 

In [4]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7782.62it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


### VectorDB

In [5]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "quantum_data", persist_directory: str = "./data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    


Vector store initialized. Collection: quantum_data
Existing documents in collection: 110


In [6]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 55 texts...


Batches: 100%|██████████| 2/2 [00:00<00:00,  4.14it/s]

Generated embeddings with shape: (55, 384)
Adding 55 documents to vector store...
Successfully added 55 documents to vector store
Total documents in collection: 165


### RAG Retrieval

In [7]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore,embedding_manager)

rag_retriever

rag_retriever.retrieve("What is quantum entanglement?", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'What is quantum entanglement?'
Top K: 3, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 152.59it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_adbe3a9b_8',
  'content': 'HOW QUANTUM ENTANGLEMENT WORKS',
  'metadata': {'content_length': 30,
   'source': 'quantum_entanglement.txt',
   'doc_index': 8},
  'similarity_score': 0.6963869035243988,
  'distance': 0.3036130964756012,
  'rank': 1},
 {'id': 'doc_040fa8c0_8',
  'content': 'HOW QUANTUM ENTANGLEMENT WORKS',
  'metadata': {'source': 'quantum_entanglement.txt',
   'content_length': 30,
   'doc_index': 8},
  'similarity_score': 0.6963869035243988,
  'distance': 0.3036130964756012,
  'rank': 2},
 {'id': 'doc_d7676b4e_8',
  'content': 'HOW QUANTUM ENTANGLEMENT WORKS',
  'metadata': {'source': 'quantum_entanglement.txt',
   'content_length': 30,
   'doc_index': 8},
  'similarity_score': 0.6963869035243988,
  'distance': 0.3036130964756012,
  'rank': 3}]

### Integrating VectorDB Context Pipeline with LLM output

In [15]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

### Initialize the Groq LLM (set your GROQ_API_KEY in environment)
groq_api_key=os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query,retriever,llm,top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    ## Generate the answer using GROQ LLM
    prompt = f"""You are QuantumAI, an AI assistant exclusively dedicated to quantum mechanics and quantum information science.

    Your knowledge scope is strictly limited to:
    - Quantum entanglement theory, history, and experimental evidence
    - Bell's theorem, Bell inequalities, and EPR paradox
    - Quantum information science: teleportation, cryptography, and computing
    - Quantum hardware: ion traps, superconducting qubits, photonic systems
    - Decoherence, entanglement entropy, and quantum error correction
    - Civil and natural occurrences of quantum phenomena (biology, cosmology)

    General politeness and professionalism in communication should be maintained, but you are not a general assistant. Your expertise is solely in quantum mechanics and quantum entanglement.

    You are NOT permitted to answer questions outside this scope under any circumstances. Do not attempt to be helpful on out-of-scope topics.

    Here is the retrieved knowledge context for this query:
    {context}

    Question: {query}

    Instructions:
    1. If the question is about a specific fact, experiment, or named concept, answer strictly using the provided context.
    2. If the question is a general quantum mechanics or physics question, use your knowledge to answer it.
    3. If combining both, answer clearly and don't hesitate to use the retrieved context to enhance your response.
    4. If the question is outside your defined scope — meaning it is not about quantum mechanics, entanglement, or related physics — politely refuse and remind the user of your area of expertise.

    Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

    # if not context:
    #     return "No relevant context found to answer the question."
    
    

In [19]:
# Now run the RAG query
answer = rag_simple("hello?", rag_retriever, llm) # This question is out of scope, so the model should politely refuse to answer and remind the user of its expertise in quantum mechanics.
print(answer)


Retrieving documents for query: 'hello?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 126.70it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


Hello. I'm QuantumAI, your dedicated assistant for quantum mechanics and quantum information science. I'm here to help with any questions you may have within my scope of expertise.

Please feel free to ask your question, and I'll do my best to provide a clear and accurate response. If your question falls outside my area of expertise, I'll politely let you know and explain why I'm unable to assist.

What's your question?


In [20]:
answer = rag_simple("what is quantum entanglement?", rag_retriever, llm) # This question is in scope, so the model should use the retrieved context to answer it accurately and comprehensively.
print(answer)

Retrieving documents for query: 'what is quantum entanglement?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 110.48it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Quantum entanglement is a fundamental concept in quantum mechanics where two or more particles become correlated in such a way that the state of one particle cannot be described independently of the others, even when they are separated by large distances.

In entangled particles, measuring the state of one particle will instantaneously affect the state of the other entangled particles, regardless of the distance between them. This phenomenon is often referred to as "spooky action at a distance" due to its seemingly instantaneous nature, which appears to defy classical notions of space and time.

To illustrate this concept, consider a pair of entangled particles, each with a spin of 1/2. If the spin of one particle is measured to be "up," the spin of the other particle will be determined to be "down," and vice versa, regardless of the distance between them. This correlation is not due to any physical connection or communication between the particles, but rather a fundamental property of